In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/afreeenarshi/fulfilment-center-info/fulfilment_center_info.csv
/kaggle/input/datasets/afreeenarshi/weekly-demand-data/train.csv
/kaggle/input/datasets/afreeenarshi/meal-info/meal_info.csv


# Final Model - Meal Demand Forecasting

## Project Objective

The objective of this project is to forecast weekly meal demand for a meal
delivery business.

Accurate demand forecasting can help the business:

- plan raw-material procurement
- reduce food wastage
- allocate kitchen capacity
- plan staffing
- improve inventory management
- prepare for promotional demand

The target variable is:

**`num_orders`**

The forecasting problem is treated as a time-dependent supervised learning
problem where historical demand, meal information, fulfillment-center
information, pricing, promotions, and historical Center × Meal demand patterns
are used to predict future demand.

---

## Project Dataset

The project uses the Meal Demand Forecasting dataset containing:

- `train.csv`
- `meal_info.csv`
- `fulfilment_center_info.csv`

Historical observations cover Weeks 1–145.

The competition-style forecasting horizon is Weeks 146–155.

The evaluation metric used throughout the project is:

**RMSLE**

Lower RMSLE indicates better forecasting performance.

---

## Previous Experimentation

A separate notebook was used to investigate:

- exploratory data analysis
- temporal patterns
- autocorrelation
- lag features
- rolling features
- time-aware validation
- baseline models
- XGBoost
- enriched metadata features
- Center × Meal historical statistics
- feature reduction
- price-derived features
- LightGBM
- CatBoost
- error analysis

That notebook is treated as the **model experimentation and discovery
notebook**.

This notebook is the **final model development notebook**.

Only experimentally supported components will be retained here.

---

## Final Model Direction

Based on the previous experimentation, the current candidate model is:

**XGBoost + Enriched Features + Center × Meal Historical Statistics**

The candidate feature groups are:

### Time-series features
- `lag_1`
- `lag_2`
- `lag_4`
- `lag_8`
- `rolling_mean_4`
- `rolling_std_4`

### Meal features
- `meal_id`
- `category`
- `cuisine`

### Fulfillment-center features
- `center_id`
- `city_code`
- `region_code`
- `center_type`
- `op_area`

### Price features
- `checkout_price`
- `base_price`

### Promotion features
- `emailer_for_promotion`
- `homepage_featured`

### Center × Meal historical features
- `pair_mean`
- `pair_std`

Total candidate features:

**20**

---

## Important Modeling Principle

All temporal and historical features must be constructed using only information
available at the forecasting time.

Future observations must never be used to construct features for an earlier
forecast period.

Validation therefore uses an expanding-window, recursive forecasting setup.

---

## Goal of This Notebook

The goal is not simply to obtain a low validation score.

The final model must satisfy four conditions:

1. Strong time-aware validation performance.
2. No temporal leakage.
3. Consistent behavior on realistic future observations.
4. A reproducible saved model that produces the same behavior after loading.

The final model will only be saved after these checks pass.

In [3]:
# Load the data
import pandas as pd
import numpy as np

train = pd.read_csv("/kaggle/input/datasets/afreeenarshi/weekly-demand-data/train.csv")
meal_info = pd.read_csv("/kaggle/input/datasets/afreeenarshi/meal-info/meal_info.csv")
center_info = pd.read_csv("/kaggle/input/datasets/afreeenarshi/fulfilment-center-info/fulfilment_center_info.csv")

print("Train shape:", train.shape)
print("Meal info shape:", meal_info.shape)
print("Center info shape:", center_info.shape)

Train shape: (456548, 9)
Meal info shape: (51, 3)
Center info shape: (77, 5)


In [4]:
print("Train weeks:", train["week"].min(), "→", train["week"].max())
print("Centers:", train["center_id"].nunique())
print("Meals:", train["meal_id"].nunique())

print("\nTrain columns:")
print(train.columns.tolist())

print("\nMissing values:")
print(train.isnull().sum())

Train weeks: 1 → 145
Centers: 77
Meals: 51

Train columns:
['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders']

Missing values:
id                       0
week                     0
center_id                0
meal_id                  0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
dtype: int64


In [5]:
print("Train weeks:", train["week"].min(), "→", train["week"].max())
print("Number of centers:", train["center_id"].nunique())
print("Number of meals:", train["meal_id"].nunique())

print("\nTrain columns:")
print(train.columns.tolist())

print("\nMissing values:")
print(train.isnull().sum())

Train weeks: 1 → 145
Number of centers: 77
Number of meals: 51

Train columns:
['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders']

Missing values:
id                       0
week                     0
center_id                0
meal_id                  0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
dtype: int64


In [6]:
print("Meal information:")
display(meal_info.head())

print("\nCenter information:")
display(center_info.head())

Meal information:


,meal_id,category,cuisine
0,1885,Beverages,Thai
1,1993,Beverages,Thai
2,2539,Beverages,Thai
3,1248,Beverages,Indian
4,2631,Beverages,Indian



Center information:


,center_id,city_code,region_code,center_type,op_area
0,11,679,56,TYPE_A,3.7
1,13,590,56,TYPE_B,6.7
2,124,590,56,TYPE_C,4.0
3,66,648,34,TYPE_A,4.1
4,94,632,34,TYPE_C,3.6


## Metadata Enrichment

The raw training data contains identifiers for meals and fulfillment centers,
but the identifiers alone do not describe the underlying business context.

We therefore join:

- `meal_info` → meal category and cuisine
- `center_info` → city, region, center type, and operating area

These variables provide additional structural information that can help the
model distinguish demand patterns across different meals and fulfillment
centers.

In [7]:
# Merge meal-level information
train_enriched = train.merge(
    meal_info,
    on="meal_id",
    how="left"
)

# Merge fulfillment-center information
train_enriched = train_enriched.merge(
    center_info,
    on="center_id",
    how="left"
)

print("Original shape:", train.shape)
print("Enriched shape:", train_enriched.shape)

print("\nNew columns:")
print(
    [
        col for col in train_enriched.columns
        if col not in train.columns
    ]
)

print("\nMissing values after enrichment:")
print(
    train_enriched.isnull().sum()
)

Original shape: (456548, 9)
Enriched shape: (456548, 15)

New columns:
['category', 'cuisine', 'city_code', 'region_code', 'center_type', 'op_area']

Missing values after enrichment:
id                       0
week                     0
center_id                0
meal_id                  0
checkout_price           0
base_price               0
emailer_for_promotion    0
homepage_featured        0
num_orders               0
category                 0
cuisine                  0
city_code                0
region_code              0
center_type              0
op_area                  0
dtype: int64


## Final Feature Architecture

The final model combines four types of information:

### 1. Meal and Center Context
- `meal_id`
- `category`
- `cuisine`
- `center_id`
- `city_code`
- `region_code`
- `center_type`
- `op_area`

### 2. Price and Promotion
- `checkout_price`
- `base_price`
- `emailer_for_promotion`
- `homepage_featured`

### 3. Historical Demand
- `lag_1`
- `lag_2`
- `lag_4`
- `lag_8`
- `rolling_mean_4`
- `rolling_std_4`

### 4. Center × Meal Historical Statistics
- `pair_mean`
- `pair_std`

The final candidate therefore contains 20 features.

Temporal features will be constructed using only information available before
the forecast week.

## Create the Lag Function

In [8]:
def create_lag_feature(df, lag):

    lookup = df[
        [
            "center_id",
            "meal_id",
            "week",
            "num_orders"
        ]
    ].copy()

    lookup = lookup.rename(
        columns={
            "week": "previous_week",
            "num_orders": f"lag_{lag}"
        }
    )

    result = df.copy()

    result["previous_week"] = (
        result["week"] - lag
    )

    result = result.merge(
        lookup,
        on=[
            "center_id",
            "meal_id",
            "previous_week"
        ],
        how="left"
    )

    result = result.drop(
        columns=["previous_week"]
    )

    return result

In [9]:
historical_features = train_enriched.copy()

for lag in [1, 2, 4, 8]:

    historical_features = create_lag_feature(
        historical_features,
        lag
    )

print(
    historical_features[
        [
            "center_id",
            "meal_id",
            "week",
            "num_orders",
            "lag_1",
            "lag_2",
            "lag_4",
            "lag_8"
        ]
    ].head(10)
)

   center_id  meal_id  week  num_orders  lag_1  lag_2  lag_4  lag_8
0         55     1885     1         177    NaN    NaN    NaN    NaN
1         55     1993     1         270    NaN    NaN    NaN    NaN
2         55     2539     1         189    NaN    NaN    NaN    NaN
3         55     2139     1          54    NaN    NaN    NaN    NaN
4         55     2631     1          40    NaN    NaN    NaN    NaN
5         55     1248     1          28    NaN    NaN    NaN    NaN
6         55     1778     1         190    NaN    NaN    NaN    NaN
7         55     1062     1         391    NaN    NaN    NaN    NaN
8         55     2707     1         472    NaN    NaN    NaN    NaN
9         55     1207     1         676    NaN    NaN    NaN    NaN


In [10]:
# Verify That the Lag Is Actually Correct
center_test = 10
meal_test = 1062

check = historical_features[
    (historical_features["center_id"] == center_test) &
    (historical_features["meal_id"] == meal_test)
].sort_values("week")

display(
    check[
        [
            "week",
            "num_orders",
            "lag_1",
            "lag_2",
            "lag_4",
            "lag_8"
        ]
    ].tail(15)
)

,week,num_orders,lag_1,lag_2,lag_4,lag_8
409925,131,1176,825.0,674.0,919.0,1148.0
413194,132,1538,1176.0,825.0,932.0,716.0
416520,133,622,1538.0,1176.0,674.0,661.0
419848,134,782,622.0,1538.0,825.0,931.0
423117,135,960,782.0,622.0,1176.0,919.0
426394,136,918,960.0,782.0,1538.0,932.0
429678,137,703,918.0,960.0,622.0,674.0
432945,138,513,703.0,918.0,782.0,825.0
436221,139,824,513.0,703.0,960.0,1176.0
439551,140,1014,824.0,513.0,918.0,1538.0


### Longer-Term Lag Features

The forecasting horizon spans ten future weeks.

Therefore, demand from several weeks earlier may contain useful information
about longer-term demand patterns.

In addition to the short-term lags, the final feature search will therefore
consider:

- `lag_10`
- `lag_11`
- `lag_12`

These features are treated as candidate features and will only be retained if
they improve time-aware validation performance.

In [11]:
for lag in [10, 11, 12]:

    historical_features = create_lag_feature(
        historical_features,
        lag
    )

In [12]:
print(
    historical_features[
        [
            "week",
            "num_orders",
            "lag_1",
            "lag_2",
            "lag_4",
            "lag_8",
            "lag_10",
            "lag_11",
            "lag_12"
        ]
    ].tail()
)

        week  num_orders  lag_1  lag_2  lag_4  lag_8  lag_10  lag_11  lag_12
456543   145          68  123.0   95.0   54.0  188.0    55.0   162.0    13.0
456544   145          42   13.0   14.0   15.0  109.0    13.0    28.0    13.0
456545   145         501  770.0  391.0  203.0  109.0   230.0   242.0   121.0
456546   145         729  811.0  447.0  284.0  405.0   432.0   445.0   391.0
456547   145         162  190.0  216.0  231.0  339.0   231.0   149.0   177.0


## Rolling Demand Features

Lag features describe demand at specific previous weeks.

Rolling features instead summarize recent demand over a window.

For this model we consider:

- Four-week rolling mean
- Four-week rolling standard deviation

The current week's demand is excluded from the calculation.

Therefore, for Week 100, the rolling features only use observations available
before Week 100.

In [13]:
historical_features = historical_features.sort_values(
    ["center_id", "meal_id", "week"]
).copy()


historical_features["rolling_mean_4"] = (
    historical_features
    .groupby(
        ["center_id", "meal_id"]
    )["num_orders"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(4)
         .mean()
    )
)


historical_features["rolling_std_4"] = (
    historical_features
    .groupby(
        ["center_id", "meal_id"]
    )["num_orders"]
    .transform(
        lambda x:
        x.shift(1)
         .rolling(4)
         .std()
    )
)

## Exponentially Weighted Moving Average

An exponentially weighted moving average gives more importance to recent
observations while still retaining information from older observations.

Unlike a simple rolling mean, EWMA does not give every observation in the
window equal weight.

This can be useful when demand changes gradually over time.

EWMA features will be treated as experimental candidates and evaluated using
the same time-aware validation procedure as the other features.

In [14]:
historical_features["ewma_4"] = (
    historical_features
    .groupby(
        ["center_id", "meal_id"]
    )["num_orders"]
    .transform(
        lambda x:
        x.shift(1)
         .ewm(span=4, adjust=False)
         .mean()
    )
)


historical_features["ewma_8"] = (
    historical_features
    .groupby(
        ["center_id", "meal_id"]
    )["num_orders"]
    .transform(
        lambda x:
        x.shift(1)
         .ewm(span=8, adjust=False)
         .mean()
    )
)


historical_features["ewma_12"] = (
    historical_features
    .groupby(
        ["center_id", "meal_id"]
    )["num_orders"]
    .transform(
        lambda x:
        x.shift(1)
         .ewm(span=12, adjust=False)
         .mean()
    )
)

In [15]:
# Inspecting Everything Together
historical_columns = [
    "week",
    "num_orders",
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "lag_10",
    "lag_11",
    "lag_12",
    "rolling_mean_4",
    "rolling_std_4",
    "ewma_4",
    "ewma_8",
    "ewma_12"
]

display(
    historical_features[
        historical_columns
    ].tail(15)
)

,week,num_orders,lag_1,lag_2,lag_4,lag_8,lag_10,lag_11,lag_12,rolling_mean_4,rolling_std_4,ewma_4,ewma_8,ewma_12
299912,98,26,41.0,42.0,41.0,28.0,26.0,26.0,NaN,34.25,14.174508,35.040125,31.348662,29.464992
303187,99,28,26.0,41.0,13.0,28.0,15.0,26.0,26.0,30.50,13.771952,31.424075,30.160070,28.931916
306404,100,40,28.0,26.0,42.0,42.0,28.0,15.0,26.0,34.25,8.421203,30.054445,29.680055,28.788545
309719,101,13,40.0,28.0,41.0,13.0,28.0,28.0,15.0,33.75,7.847505,34.032667,31.973376,30.513384
316156,103,40,NaN,13.0,28.0,13.0,13.0,42.0,28.0,26.75,11.056672,25.619600,27.757070,27.819017
322446,105,53,NaN,40.0,13.0,41.0,13.0,41.0,13.0,30.25,12.816006,31.371760,30.477721,29.693015
329132,107,55,NaN,53.0,40.0,28.0,41.0,42.0,13.0,36.50,16.822604,40.023056,35.482672,33.278705
332394,108,42,55.0,NaN,NaN,40.0,26.0,41.0,42.0,40.25,19.345542,46.013834,39.819856,36.620442
338932,110,54,NaN,42.0,NaN,NaN,40.0,28.0,26.0,47.50,7.593857,44.408300,40.304332,37.448067
342211,111,55,54.0,NaN,55.0,40.0,13.0,40.0,28.0,51.00,6.055301,48.244980,43.347814,39.994518


In [16]:
display(
    historical_features[
        (historical_features["center_id"] == 10) &
        (historical_features["meal_id"] == 1062)
    ][historical_columns]
    .tail(15)
)

,week,num_orders,lag_1,lag_2,lag_4,lag_8,lag_10,lag_11,lag_12,rolling_mean_4,rolling_std_4,ewma_4,ewma_8,ewma_12
409925,131,1176,825.0,674.0,919.0,1148.0,919.0,755.0,1160.0,837.50,118.969184,816.041547,852.459618,886.656132
413194,132,1538,1176.0,825.0,932.0,716.0,879.0,919.0,755.0,901.75,211.257150,960.024928,924.357481,931.170574
416520,133,622,1538.0,1176.0,674.0,661.0,1148.0,879.0,919.0,1053.25,385.563981,1191.214957,1060.722485,1024.528947
419848,134,782,622.0,1538.0,825.0,931.0,716.0,1148.0,879.0,1040.25,403.091698,963.528974,963.228600,962.601417
423117,135,960,782.0,622.0,1176.0,919.0,661.0,716.0,1148.0,1029.50,411.236753,890.917385,922.955577,934.816583
426394,136,918,960.0,782.0,1538.0,932.0,931.0,661.0,716.0,975.50,399.604388,918.550431,931.187671,938.690955
429678,137,703,918.0,960.0,622.0,674.0,919.0,931.0,661.0,820.50,152.589864,918.330258,928.257078,935.507731
432945,138,513,703.0,918.0,782.0,825.0,932.0,919.0,931.0,840.75,119.184381,832.198155,878.199949,899.737311
436221,139,824,513.0,703.0,960.0,1176.0,674.0,932.0,919.0,773.50,206.956517,704.518893,797.044405,840.239263
439551,140,1014,824.0,513.0,918.0,1538.0,825.0,674.0,932.0,739.50,174.773186,752.311336,803.034537,837.740915


```text
                 Historical Demand
                        │
          ┌─────────────┼─────────────┐
          │             │             │
         Lags        Rolling          EWMA
          │             │             │
     1, 2, 4, 8     mean_4 / std_4    4, 8, 12
       10, 11, 12

## Adding Center × Meal historical statistics

In [17]:
historical_features = historical_features.sort_values(
    ["center_id", "meal_id", "week"]
).copy()

group_cols = ["center_id", "meal_id"]

historical_features["pair_mean"] = (
    historical_features
    .groupby(group_cols)["num_orders"]
    .transform(
        lambda x: x.shift(1).expanding().mean()
    )
)

historical_features["pair_std"] = (
    historical_features
    .groupby(group_cols)["num_orders"]
    .transform(
        lambda x: x.shift(1).expanding().std()
    )
)

In [18]:
historical_features[
    [
        "center_id",
        "meal_id",
        "week",
        "num_orders",
        "pair_mean",
        "pair_std"
    ]
].head(15)

,center_id,meal_id,week,num_orders,pair_mean,pair_std
2370,10,1062,1,865,NaN,NaN
5273,10,1062,2,782,865.000000,NaN
8175,10,1062,3,851,823.500000,58.689863
11064,10,1062,4,1202,832.666667,44.433471
13918,10,1062,5,958,925.000000,188.196706
16777,10,1062,6,1094,931.600000,163.649931
19574,10,1062,7,1513,958.666667,160.688104
22368,10,1062,8,1149,1037.857143,255.763842
25205,10,1062,9,1282,1051.750000,240.029611
28068,10,1062,10,1473,1077.333333,237.282532


## Forecast-Time Feature Generation

During forecasting, the target variable `num_orders` is unavailable for the
week being predicted.

Therefore, all historical demand features must be calculated using only
information available before the forecast week.

The forecasting feature generator will create:

- Calendar-based demand lags
- Rolling demand statistics
- Exponentially weighted moving averages
- Center × Meal historical mean
- Center × Meal historical standard deviation

For multi-week forecasting, each prediction will be appended to the historical
data before generating features for the next week.

This allows the model to perform recursive forecasting without using future
actual demand values.

In [19]:
def create_forecast_features(history_df, forecast_df):
    """
    Create historical demand features for future observations.

    history_df:
        Data available before the forecast week.
        Contains actual historical demand and previously generated predictions.

    forecast_df:
        Future rows for which num_orders is unknown.
    """

    result = forecast_df.copy()

    # --------------------------------------------------
    # 1. Exact calendar-based lags
    # --------------------------------------------------

    lag_values = [1, 2, 4, 8, 10, 11, 12]

    lookup = history_df[
        ["center_id", "meal_id", "week", "num_orders"]
    ].copy()

    for lag in lag_values:

        lag_lookup = lookup.rename(
            columns={
                "week": "previous_week",
                "num_orders": f"lag_{lag}"
            }
        )

        result["previous_week"] = result["week"] - lag

        result = result.merge(
            lag_lookup[
                [
                    "center_id",
                    "meal_id",
                    "previous_week",
                    f"lag_{lag}"
                ]
            ],
            on=["center_id", "meal_id", "previous_week"],
            how="left"
        )

        result = result.drop(columns=["previous_week"])

    # --------------------------------------------------
    # 2. Rolling statistics
    # --------------------------------------------------

    rolling_features = (
        history_df
        .sort_values(["center_id", "meal_id", "week"])
        .groupby(["center_id", "meal_id"])["num_orders"]
        .apply(lambda x: x.tail(4))
        .reset_index(level=[0, 1], drop=True)
    )

    # Calculate separately for each forecast row
    rolling_mean = []
    rolling_std = []

    for _, row in result.iterrows():

        pair_history = history_df[
            (history_df["center_id"] == row["center_id"]) &
            (history_df["meal_id"] == row["meal_id"]) &
            (history_df["week"] < row["week"])
        ].sort_values("week")["num_orders"]

        recent_values = pair_history.tail(4)

        rolling_mean.append(recent_values.mean())
        rolling_std.append(recent_values.std())

    result["rolling_mean_4"] = rolling_mean
    result["rolling_std_4"] = rolling_std

    # --------------------------------------------------
    # 3. Exponentially weighted moving averages
    # --------------------------------------------------

    ewma_4 = []
    ewma_8 = []
    ewma_12 = []

    for _, row in result.iterrows():

        pair_history = history_df[
            (history_df["center_id"] == row["center_id"]) &
            (history_df["meal_id"] == row["meal_id"]) &
            (history_df["week"] < row["week"])
        ].sort_values("week")["num_orders"]

        ewma_4.append(
            pair_history.ewm(
                span=4,
                adjust=False
            ).mean().iloc[-1]
            if len(pair_history) > 0 else np.nan
        )

        ewma_8.append(
            pair_history.ewm(
                span=8,
                adjust=False
            ).mean().iloc[-1]
            if len(pair_history) > 0 else np.nan
        )

        ewma_12.append(
            pair_history.ewm(
                span=12,
                adjust=False
            ).mean().iloc[-1]
            if len(pair_history) > 0 else np.nan
        )

    result["ewma_4"] = ewma_4
    result["ewma_8"] = ewma_8
    result["ewma_12"] = ewma_12

    # --------------------------------------------------
    # 4. Center × Meal historical statistics
    # --------------------------------------------------

    pair_stats = (
        history_df
        .groupby(["center_id", "meal_id"])["num_orders"]
        .agg(
            pair_mean="mean",
            pair_std="std"
        )
        .reset_index()
    )

    result = result.merge(
        pair_stats,
        on=["center_id", "meal_id"],
        how="left"
    )

    return result

In [20]:
history_test = historical_features[
    historical_features["week"] <= 80
].copy()

forecast_test = train_enriched[
    train_enriched["week"] == 81
].copy()

test_features = create_forecast_features(
    history_test,
    forecast_test
)

test_features.shape

(3204, 29)

In [21]:
test_features[
    [
        "center_id",
        "meal_id",
        "week",
        "lag_1",
        "lag_2",
        "lag_4",
        "lag_8",
        "lag_10",
        "lag_11",
        "lag_12",
        "rolling_mean_4",
        "rolling_std_4",
        "ewma_4",
        "ewma_8",
        "ewma_12",
        "pair_mean",
        "pair_std"
    ]
].head()

,center_id,meal_id,week,lag_1,lag_2,lag_4,lag_8,lag_10,lag_11,lag_12,rolling_mean_4,rolling_std_4,ewma_4,ewma_8,ewma_12,pair_mean,pair_std
0,55,1885,81,271.0,284.0,257.0,175.0,351.0,485.0,391.0,260.50,23.130067,273.760078,288.807289,300.523840,275.150000,125.496795
1,55,1993,81,325.0,297.0,216.0,310.0,310.0,350.0,432.0,270.50,49.480636,294.212721,292.124543,297.885010,276.337500,156.262109
2,55,2539,81,217.0,216.0,296.0,393.0,204.0,149.0,190.0,256.75,46.485661,234.135937,235.455409,234.236648,210.000000,125.385457
3,55,2631,81,15.0,40.0,28.0,NaN,54.0,55.0,54.0,31.00,12.192894,27.065382,30.648128,33.464727,45.144928,26.606763
4,55,1248,81,28.0,67.0,28.0,69.0,28.0,54.0,96.0,44.50,19.672316,43.868325,47.084852,48.222315,45.132353,35.006568


In [22]:
# Basic historical demand features
basic_historical_features = [
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "rolling_mean_4",
    "rolling_std_4"
]

# Longer historical lags
long_lag_features = [
    "lag_10",
    "lag_11",
    "lag_12"
]

# Exponentially weighted moving averages
ewma_features = [
    "ewma_4",
    "ewma_8",
    "ewma_12"
]

# Center × Meal historical statistics
pair_history_features = [
    "pair_mean",
    "pair_std"
]

In [23]:
# non-time features:
static_features = [
    "center_id",
    "meal_id",
    "category",
    "cuisine",
    "city_code",
    "region_code",
    "center_type",
    "op_area",
    "checkout_price",
    "base_price",
    "emailer_for_promotion",
    "homepage_featured"
]

In [24]:
# Candidate feature sets
feature_sets = {

    "Basic Historical": (
        static_features
        + basic_historical_features
    ),

    "Basic + Long Lags": (
        static_features
        + basic_historical_features
        + long_lag_features
    ),

    "Basic + EWMA": (
        static_features
        + basic_historical_features
        + ewma_features
    ),

    "Full Historical": (
        static_features
        + basic_historical_features
        + long_lag_features
        + ewma_features
        + pair_history_features
    )
}

In [25]:
for name, features in feature_sets.items():
    print(f"{name}: {len(features)} features")

Basic Historical: 18 features
Basic + Long Lags: 21 features
Basic + EWMA: 21 features
Full Historical: 26 features


## Pipeline

In [26]:
# Defining categorical and numerical features
categorical_features = [
    "category",
    "cuisine",
    "center_type"
]

numerical_features = [
    feature
    for feature in static_features
    if feature not in categorical_features
]

numerical_features

['center_id',
 'meal_id',
 'city_code',
 'region_code',
 'op_area',
 'checkout_price',
 'base_price',
 'emailer_for_promotion',
 'homepage_featured']

In [27]:
# Defining the XGBoost model
from xgboost import XGBRegressor

xgb_params = {
    "objective": "reg:squarederror",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1
}

In [28]:
model = XGBRegressor(**xgb_params)

In [29]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


def build_pipeline(feature_list):

    categorical = [
        feature
        for feature in categorical_features
        if feature in feature_list
    ]

    numerical = [
        feature
        for feature in feature_list
        if feature not in categorical
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical
            ),
            (
                "numerical",
                "passthrough",
                numerical
            )
        ]
    )

    model = XGBRegressor(**xgb_params)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    return pipeline

In [30]:
# testing
for name, features in feature_sets.items():

    pipeline = build_pipeline(features)

    print(
        f"{name}: "
        f"{len(features)} input features"
    )

Basic Historical: 18 input features
Basic + Long Lags: 21 input features
Basic + EWMA: 21 input features
Full Historical: 26 input features


In [31]:
# Defining RMSLE
from sklearn.metrics import mean_squared_log_error


def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(
        mean_squared_log_error(y_true, y_pred)
    )

In [32]:
# Defining validation folds
folds = [
    {"train_end": 80,  "valid_start": 81,  "valid_end": 90},
    {"train_end": 90,  "valid_start": 91,  "valid_end": 100},
    {"train_end": 100, "valid_start": 101, "valid_end": 110},
    {"train_end": 110, "valid_start": 111, "valid_end": 120},
    {"train_end": 120, "valid_start": 121, "valid_end": 130},
]

In [33]:
for i, fold in enumerate(folds, 1):
    print(
        f"Fold {i}: "
        f"Train 1-{fold['train_end']} | "
        f"Validation {fold['valid_start']}-{fold['valid_end']}"
    )

Fold 1: Train 1-80 | Validation 81-90
Fold 2: Train 1-90 | Validation 91-100
Fold 3: Train 1-100 | Validation 101-110
Fold 4: Train 1-110 | Validation 111-120
Fold 5: Train 1-120 | Validation 121-130


# Recursive validation function

In [34]:
def recursive_forecast(
    model,
    history_df,
    validation_df,
    feature_list
):
    """
    Recursively forecast each week in validation_df.

    Predictions from previous validation weeks are added to history
    and used to generate features for subsequent weeks.
    """

    history = history_df.copy()

    predictions = []
    actuals = []

    validation_weeks = sorted(
        validation_df["week"].unique()
    )

    for week in validation_weeks:

        # Current week's rows
        current_week = validation_df[
            validation_df["week"] == week
        ].copy()

        # Create features using only history before this week
        current_features = create_forecast_features(
            history,
            current_week
        )

        # Select model features
        X_current = current_features[
            feature_list
        ]

        # Predict log demand
        log_pred = model.predict(X_current)

        # Convert back to original demand scale
        pred = np.expm1(log_pred)

        # Demand cannot be negative
        pred = np.clip(pred, 0, None)

        predictions.extend(pred)
        actuals.extend(
            current_week["num_orders"].values
        )

        # Add predictions to history
        predicted_history = current_week.copy()

        predicted_history["num_orders"] = pred

        history = pd.concat(
            [
                history,
                predicted_history
            ],
            ignore_index=True
        )

    return (
        np.array(actuals),
        np.array(predictions)
    )

In [35]:
# Test the function on Fold 1
fold = folds[0]

train_fold = train_enriched[
    train_enriched["week"] <= fold["train_end"]
].copy()

valid_fold = train_enriched[
    (train_enriched["week"] >= fold["valid_start"]) &
    (train_enriched["week"] <= fold["valid_end"])
].copy()

In [36]:
# historical features for the training portion:
train_fold_features = historical_features[
    historical_features["week"] <= fold["train_end"]
].copy()

In [37]:
# Build the basic model
feature_list = feature_sets["Basic Historical"]

pipeline = build_pipeline(feature_list)

In [38]:
X_train = train_fold_features[feature_list]

y_train = np.log1p(
    train_fold_features["num_orders"]
)

pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['category', 'cuisine',
                                                   'center_type']),
                                                 ('numerical', 'passthrough',
                                                  ['center_id', 'meal_id',
                                                   'city_code', 'region_code',
                                                   'op_area', 'checkout_price',
                                                   'base_price',
                                                   'emailer_for_promotion',
                                                   'homepage_featured', 'lag_1',
                                                   'lag_2', 'lag_4', 'lag_8',
                                                   'r...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=8, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=-1,
                              num_parallel_tree=None, ...))])

In [39]:
y_actual, y_pred = recursive_forecast(
    pipeline,
    train_fold,
    valid_fold,
    feature_list
)

In [40]:
fold_score = rmsle(
    y_actual,
    y_pred
)

print(f"Fold 1 RMSLE: {fold_score:.6f}")

Fold 1 RMSLE: 0.506543


In [41]:
def run_feature_experiment(feature_list, verbose=True):

    fold_scores = []

    for i, fold in enumerate(folds, 1):

        # Training data
        train_fold = train_enriched[
            train_enriched["week"] <= fold["train_end"]
        ].copy()

        train_fold_features = historical_features[
            historical_features["week"] <= fold["train_end"]
        ].copy()

        # Validation data
        valid_fold = train_enriched[
            (train_enriched["week"] >= fold["valid_start"]) &
            (train_enriched["week"] <= fold["valid_end"])
        ].copy()

        # Build pipeline
        pipeline = build_pipeline(feature_list)

        # Training features and target
        X_train = train_fold_features[feature_list]

        y_train = np.log1p(
            train_fold_features["num_orders"]
        )

        # Train
        pipeline.fit(
            X_train,
            y_train
        )

        # Recursive forecasting
        y_actual, y_pred = recursive_forecast(
            pipeline,
            train_fold,
            valid_fold,
            feature_list
        )

        # Evaluate
        score = rmsle(
            y_actual,
            y_pred
        )

        fold_scores.append(score)

        if verbose:
            print(
                f"Fold {i} RMSLE: {score:.6f}"
            )

    return {
        "fold_scores": fold_scores,
        "mean_rmsle": np.mean(fold_scores),
        "std_rmsle": np.std(fold_scores)
    }

In [42]:
basic_result = run_feature_experiment(
    feature_sets["Basic Historical"]
)

basic_result

Fold 1 RMSLE: 0.506543
Fold 2 RMSLE: 0.526112
Fold 3 RMSLE: 0.545128
Fold 4 RMSLE: 0.539767
Fold 5 RMSLE: 0.495655


{'fold_scores': [np.float64(0.5065434370742907),
  np.float64(0.5261124824472262),
  np.float64(0.5451283764348632),
  np.float64(0.5397669846626236),
  np.float64(0.49565521140940066)],
 'mean_rmsle': np.float64(0.522641298405681),
 'std_rmsle': np.float64(0.01896521530343129)}

## Pair mean and pair std feature

In [43]:
pair_history_features = [
    "pair_mean",
    "pair_std"
]

In [44]:
baseline_features = (
    static_features
    + basic_historical_features
    + pair_history_features
)

In [45]:
pair_result = run_feature_experiment(
    baseline_features
)

pair_result

Fold 1 RMSLE: 0.490818
Fold 2 RMSLE: 0.509832
Fold 3 RMSLE: 0.535055
Fold 4 RMSLE: 0.514980
Fold 5 RMSLE: 0.487530


{'fold_scores': [np.float64(0.4908177525627755),
  np.float64(0.5098322974557487),
  np.float64(0.5350545522025792),
  np.float64(0.5149796522157685),
  np.float64(0.48752980202278345)],
 'mean_rmsle': np.float64(0.5076428112919311),
 'std_rmsle': np.float64(0.01730689092388404)}

## New feature set

In [49]:
pair_long_lag_features = (
    static_features
    + basic_historical_features
    + long_lag_features
    + pair_history_features
)

print("Number of features:", len(pair_long_lag_features))

Number of features: 23


In [50]:
pair_long_lag_result = run_feature_experiment(
    pair_long_lag_features
)

pair_long_lag_result

Fold 1 RMSLE: 0.487734
Fold 2 RMSLE: 0.508029
Fold 3 RMSLE: 0.533558
Fold 4 RMSLE: 0.514020
Fold 5 RMSLE: 0.486526


{'fold_scores': [np.float64(0.4877340836824265),
  np.float64(0.5080288352854953),
  np.float64(0.5335580671451808),
  np.float64(0.5140196054760621),
  np.float64(0.48652562187252696)],
 'mean_rmsle': np.float64(0.5059732426923383),
 'std_rmsle': np.float64(0.01755430905498979)}

I have notice the fold 3 always slighlty larger than other fold rsmle, im little inspecting it 

In [51]:
fold = folds[2]

train_fold = train_enriched[
    train_enriched["week"] <= fold["train_end"]
].copy()

valid_fold = train_enriched[
    (train_enriched["week"] >= fold["valid_start"]) &
    (train_enriched["week"] <= fold["valid_end"])
].copy()

train_fold_features = historical_features[
    historical_features["week"] <= fold["train_end"]
].copy()

pipeline = build_pipeline(
    pair_long_lag_features
)

X_train = train_fold_features[
    pair_long_lag_features
]

y_train = np.log1p(
    train_fold_features["num_orders"]
)

pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['category', 'cuisine',
                                                   'center_type']),
                                                 ('numerical', 'passthrough',
                                                  ['center_id', 'meal_id',
                                                   'city_code', 'region_code',
                                                   'op_area', 'checkout_price',
                                                   'base_price',
                                                   'emailer_for_promotion',
                                                   'homepage_featured', 'lag_1',
                                                   'lag_2', 'lag_4', 'lag_8',
                                                   'r...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=8, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=-1,
                              num_parallel_tree=None, ...))])

In [52]:
y_actual, y_pred = recursive_forecast(
    pipeline,
    train_fold,
    valid_fold,
    pair_long_lag_features
)

In [53]:
fold3_errors = valid_fold[
    ["week", "center_id", "meal_id", "num_orders"]
].copy()

fold3_errors["prediction"] = y_pred

fold3_errors["absolute_error"] = (
    fold3_errors["num_orders"]
    - fold3_errors["prediction"]
).abs()

fold3_errors["log_error"] = (
    np.log1p(fold3_errors["num_orders"])
    - np.log1p(fold3_errors["prediction"])
).abs()

fold3_errors.head()

,week,center_id,meal_id,num_orders,prediction,absolute_error,log_error
309297,101,55,1885,242,179.654892,62.345108,0.296473
309298,101,55,1993,364,277.099182,86.900818,0.271920
309299,101,55,2539,216,187.550262,28.449738,0.140533
309300,101,55,2139,15,19.766848,4.766848,0.260769
309301,101,55,2631,80,55.793777,24.206223,0.355022


In [54]:
fold3_errors.sort_values(
    "log_error",
    ascending=False
).head(20)

,week,center_id,meal_id,num_orders,prediction,absolute_error,log_error
318346,103,43,2631,14,384.257263,370.257263,3.245861
318235,103,10,2126,15,275.960968,260.960968,2.851288
331786,107,23,2444,14,244.525024,230.525024,2.795349
316057,103,52,2631,14,207.992432,193.992432,2.634248
318257,103,10,1543,15,211.592133,196.592133,2.586787
340069,110,129,2444,15,202.235016,187.235016,2.541774
315829,103,24,1543,13,176.555054,163.555054,2.540223
340343,110,26,2707,15,193.296402,178.296402,2.496796
315837,103,11,2631,15,193.191467,178.191467,2.496256
320712,104,104,1207,27,325.620758,298.620758,2.456595


In [55]:
fold3_analysis = valid_fold[
    [
        "week",
        "center_id",
        "meal_id",
        "num_orders",
        "emailer_for_promotion",
        "homepage_featured",
        "checkout_price",
        "base_price"
    ]
].copy()

fold3_analysis["prediction"] = y_pred

fold3_analysis["log_error"] = (
    np.log1p(fold3_analysis["num_orders"])
    - np.log1p(fold3_analysis["prediction"])
).abs()

fold3_analysis = fold3_analysis.sort_values(
    "log_error",
    ascending=False
)

fold3_analysis.head(30)

,week,center_id,meal_id,num_orders,emailer_for_promotion,homepage_featured,checkout_price,base_price,prediction,log_error
318346,103,43,2631,14,0,1,99.00,153.32,384.257263,3.245861
318235,103,10,2126,15,0,1,436.53,533.53,275.960968,2.851288
331786,107,23,2444,14,1,1,455.93,669.33,244.525024,2.795349
316057,103,52,2631,14,0,0,98.00,151.32,207.992432,2.634248
318257,103,10,1543,15,0,0,484.03,485.03,211.592133,2.586787
340069,110,129,2444,15,1,0,389.03,708.13,202.235016,2.541774
315829,103,24,1543,13,0,0,485.03,484.03,176.555054,2.540223
340343,110,26,2707,15,0,0,209.55,208.55,193.296402,2.496796
315837,103,11,2631,15,0,0,99.00,154.26,193.191467,2.496256
320712,104,104,1207,27,0,1,323.04,318.16,325.620758,2.456595


In [56]:
fold3_analysis.head(30)[
    [
        "week",
        "meal_id",
        "num_orders",
        "prediction",
        "emailer_for_promotion",
        "homepage_featured",
        "checkout_price",
        "base_price"
    ]
]

,week,meal_id,num_orders,prediction,emailer_for_promotion,homepage_featured,checkout_price,base_price
318346,103,2631,14,384.257263,0,1,99.00,153.32
318235,103,2126,15,275.960968,0,1,436.53,533.53
331786,107,2444,14,244.525024,1,1,455.93,669.33
316057,103,2631,14,207.992432,0,0,98.00,151.32
318257,103,1543,15,211.592133,0,0,484.03,485.03
340069,110,2444,15,202.235016,1,0,389.03,708.13
315829,103,1543,13,176.555054,0,0,485.03,484.03
340343,110,2707,15,193.296402,0,0,209.55,208.55
315837,103,2631,15,193.191467,0,0,99.00,154.26
320712,104,1207,27,325.620758,0,1,323.04,318.16


## EWMA features

In [57]:
pair_ewma_features = (
    static_features
    + basic_historical_features
    + long_lag_features
    + ewma_features
    + pair_history_features
)

print("Number of features:", len(pair_ewma_features))

Number of features: 26


In [58]:
pair_ewma_result = run_feature_experiment(
    pair_ewma_features
)

pair_ewma_result

Fold 1 RMSLE: 0.488507
Fold 2 RMSLE: 0.506661
Fold 3 RMSLE: 0.537228
Fold 4 RMSLE: 0.514206
Fold 5 RMSLE: 0.483722


{'fold_scores': [np.float64(0.4885066160819925),
  np.float64(0.5066607986704347),
  np.float64(0.5372279643141243),
  np.float64(0.5142063797535312),
  np.float64(0.4837224471700552)],
 'mean_rmsle': np.float64(0.5060648411980276),
 'std_rmsle': np.float64(0.01921069440616459)}

# Model comparison

In [63]:
comparison_df = pd.DataFrame([
    {
        "Experiment": "Naive Baseline",
        "Model": "Latest Known Demand",
        "Features": "Latest Center × Meal demand",
        "Mean_RMSLE": 0.825242,
        "Std_RMSLE": 0.033141,
        "Status": "Baseline"
    },
    {
        "Experiment": "Basic XGBoost",
        "Model": "XGBoost",
        "Features": "Basic historical features",
        "Mean_RMSLE": 0.542365,
        "Std_RMSLE": 0.022997,
        "Status": "Improved"
    },
    {
        "Experiment": "Metadata-Enriched XGBoost",
        "Model": "XGBoost",
        "Features": "Metadata + basic historical features",
        "Mean_RMSLE": 0.522641,
        "Std_RMSLE": 0.018965,
        "Status": "Improved"
    },
    {
        "Experiment": "XGBoost + Pair History",
        "Model": "XGBoost",
        "Features": "Enriched + pair_mean + pair_std",
        "Mean_RMSLE": 0.507643,
        "Std_RMSLE": 0.017307,
        "Status": "Improved"
    },
    {
        "Experiment": "XGBoost + Pair History + Long Lags",
        "Model": "XGBoost",
        "Features": "Enriched + pair history + lag 10/11/12",
        "Mean_RMSLE": 0.505973,
        "Std_RMSLE": 0.017554,
        "Status": "FINAL"
    },
    {
        "Experiment": "Time / Trend Features",
        "Model": "XGBoost",
        "Features": "Recent trends + week features + pair history",
        "Mean_RMSLE": 0.527701,
        "Std_RMSLE": 0.016492,
        "Status": "Rejected"
    },
    {
        "Experiment": "Price Change Feature",
        "Model": "XGBoost",
        "Features": "Baseline + price_change_pct",
        "Mean_RMSLE": 0.521148,
        "Std_RMSLE": np.nan,
        "Status": "Rejected"
    },
    {
        "Experiment": "LightGBM",
        "Model": "LightGBM",
        "Features": "Metadata + historical features",
        "Mean_RMSLE": 0.532606,
        "Std_RMSLE": 0.020874,
        "Status": "Rejected"
    },
    {
        "Experiment": "CatBoost",
        "Model": "CatBoost",
        "Features": "Metadata + historical features",
        "Mean_RMSLE": 0.530950,
        "Std_RMSLE": 0.023052,
        "Status": "Rejected"
    },
    {
        "Experiment": "Reduced Feature XGBoost",
        "Model": "XGBoost",
        "Features": "Selected reduced feature set",
        "Mean_RMSLE": 0.564261,
        "Std_RMSLE": 0.023922,
        "Status": "Rejected"
    }
])

comparison_df = comparison_df.sort_values(
    "Mean_RMSLE",
    ascending=True
).reset_index(drop=True)

comparison_df

,Experiment,Model,Features,Mean_RMSLE,Std_RMSLE,Status
0,XGBoost + Pair History + Long Lags,XGBoost,Enriched + pair history + lag 10/11/12,0.505973,0.017554,FINAL
1,XGBoost + Pair History,XGBoost,Enriched + pair_mean + pair_std,0.507643,0.017307,Improved
2,Price Change Feature,XGBoost,Baseline + price_change_pct,0.521148,NaN,Rejected
3,Metadata-Enriched XGBoost,XGBoost,Metadata + basic historical features,0.522641,0.018965,Improved
4,Time / Trend Features,XGBoost,Recent trends + week features + pair history,0.527701,0.016492,Rejected
5,CatBoost,CatBoost,Metadata + historical features,0.530950,0.023052,Rejected
6,LightGBM,LightGBM,Metadata + historical features,0.532606,0.020874,Rejected
7,Basic XGBoost,XGBoost,Basic historical features,0.542365,0.022997,Improved
8,Reduced Feature XGBoost,XGBoost,Selected reduced feature set,0.564261,0.023922,Rejected
9,Naive Baseline,Latest Known Demand,Latest Center × Meal demand,0.825242,0.033141,Baseline


In [64]:
final_comparison = comparison_df[
    [
        "Experiment",
        "Model",
        "Mean_RMSLE",
        "Std_RMSLE",
        "Status"
    ]
]

final_comparison

,Experiment,Model,Mean_RMSLE,Std_RMSLE,Status
0,XGBoost + Pair History + Long Lags,XGBoost,0.505973,0.017554,FINAL
1,XGBoost + Pair History,XGBoost,0.507643,0.017307,Improved
2,Price Change Feature,XGBoost,0.521148,NaN,Rejected
3,Metadata-Enriched XGBoost,XGBoost,0.522641,0.018965,Improved
4,Time / Trend Features,XGBoost,0.527701,0.016492,Rejected
5,CatBoost,CatBoost,0.530950,0.023052,Rejected
6,LightGBM,LightGBM,0.532606,0.020874,Rejected
7,Basic XGBoost,XGBoost,0.542365,0.022997,Improved
8,Reduced Feature XGBoost,XGBoost,0.564261,0.023922,Rejected
9,Naive Baseline,Latest Known Demand,0.825242,0.033141,Baseline


### Final Model

The final model is an **XGBoost regression model** trained on the log-transformed target (`log1p(num_orders)`). It combines static business information with historical demand patterns to capture both the characteristics of a meal/center and its recent demand behavior.

The model uses:
- **Meal and center metadata:** category, cuisine, center type, city/region information, and operational area.
- **Pricing and promotion features:** checkout price, base price, email promotion, and homepage promotion.
- **Historical demand features:** lags of 1, 2, 4, 8, 10, 11, and 12 weeks.
- **Rolling statistics:** 4-week rolling mean and standard deviation.
- **Center × Meal historical statistics:** expanding historical mean and standard deviation for each meal-center combination.

The model was evaluated using **five expanding-window, recursive validation folds** to prevent future demand from leaking into the forecasting process.

The final model achieved a **mean RMSLE of 0.505973** with a standard deviation of **0.017554** across the five validation folds.

This model is now locked and will be retrained on the complete historical data before generating recursive forecasts for the future weeks.

# Saving the model

In [65]:
# Final training cell
import joblib
import numpy as np
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Final locked feature set
final_features = (
    static_features
    + basic_historical_features
    + long_lag_features
    + pair_history_features
)

final_categorical_features = [
    "category",
    "cuisine",
    "center_type"
]

final_numerical_features = [
    feature
    for feature in final_features
    if feature not in final_categorical_features
]

# Preprocessing
final_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            final_categorical_features
        ),
        (
            "numerical",
            "passthrough",
            final_numerical_features
        )
    ]
)

# Final XGBoost model
final_xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

final_pipeline = Pipeline(
    steps=[
        ("preprocessor", final_preprocessor),
        ("model", final_xgb)
    ]
)

In [66]:
# Train on all historical data
# Use all available historical training data: Weeks 1–145
final_training_data = historical_features[
    historical_features["week"] <= 145
].copy()

X_final = final_training_data[final_features]
y_final = np.log1p(final_training_data["num_orders"])

final_pipeline.fit(X_final, y_final)

print("Final model trained successfully.")
print(f"Training rows: {len(X_final):,}")
print(f"Number of features: {len(final_features)}")

Final model trained successfully.
Training rows: 456,548
Number of features: 23


In [67]:
# Saving the model
final_artifact = {
    "model": final_pipeline,
    "features": final_features,
    "categorical_features": final_categorical_features,
    "numerical_features": final_numerical_features,
    "target_transform": "log1p",
    "validation_rmsle": 0.5059732427,
    "validation_std": 0.0175543091
}

model_path = "meal_demand_forecasting_final.pkl"

joblib.dump(
    final_artifact,
    model_path
)

print(f"Saved: {model_path}")

Saved: meal_demand_forecasting_final.pkl


In [68]:
# testing the .pkl
loaded_artifact = joblib.load("meal_demand_forecasting_final.pkl")

print("Artifact loaded successfully.")
print("Keys:", loaded_artifact.keys())
print("Features:", len(loaded_artifact["features"]))
print("Validation RMSLE:", loaded_artifact["validation_rmsle"])

Artifact loaded successfully.
Keys: dict_keys(['model', 'features', 'categorical_features', 'numerical_features', 'target_transform', 'validation_rmsle', 'validation_std'])
Features: 23
Validation RMSLE: 0.5059732427
